In [92]:
import pandas as pd
import numpy as np

In [100]:
import matplotlib.pyplot as plt
from PIL import Image

In [114]:
import os
import pathlib
from tqdm import tqdm

In [108]:
df_reid_cropped = pd.read_csv('../data/processed/df_cat_reid_cropped.csv')
df_reid_cropped_modified = df_reid_cropped.copy()

In [109]:
PATH_LABELS = "./runs/data/interim/det_26m/labels"
labels = [os.path.basename(f).split('.')[0] for f in os.scandir(PATH_LABELS) if os.path.basename(f).endswith('txt')]

In [117]:
def crop_yolo_detections(df, label_dir, output_dir, padding=1.1):
    base_out_path = pathlib.Path(output_dir)
    
    for _, row in tqdm(df.iterrows()):
        img_path = pathlib.Path(row['file_path'])
        txt_path = pathlib.Path(label_dir) / f"{row['file_name']}.txt"
        
        if not img_path.exists() or not txt_path.exists():
            continue
            
        data = np.loadtxt(txt_path)
        if data.ndim != 1:
            continue
            
        sub_out_path = base_out_path / str(row['dir_name'])
        sub_out_path.mkdir(parents=True, exist_ok=True)
        
        with Image.open(img_path) as img:
            w, h = img.size
            xc, yc, bw, bh = data[1:5]
            
            bw_pad = bw * padding
            bh_pad = bh * padding
            
            xmin = int((xc - bw_pad / 2) * w)
            ymin = int((yc - bh_pad / 2) * h)
            xmax = int((xc + bw_pad / 2) * w)
            ymax = int((yc + bh_pad / 2) * h)
            
            cropped = img.crop((max(0, xmin), max(0, ymin), min(w, xmax), min(h, ymax)))
            cropped.save(sub_out_path / f"{row['file_name']}.jpg")

In [118]:
crop_yolo_detections(df=df_reid_cropped_modified, label_dir=PATH_LABELS, output_dir='../data/processed/cropped')

12872it [10:19, 20.79it/s] 
